In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import time

In [ ]:
df = pd.read_csv("../Data/2_lab_proc.csv", index_col=False)

In [ ]:
df.head()

In [ ]:
df.info(verbose=True, show_counts=True)

In [ ]:
df.describe()

## Мусорные данные Header image

In [ ]:
df["header_image"]

In [ ]:
df = df.drop(["header_image"], axis=1)

## Мусорные данные Name, Reviews

Удаляю Name, уже сделал embending

In [ ]:
df = df.drop(labels=["name"], axis=1)

In [ ]:
df["reviews"].value_counts()

#### Лучше удалить, так как дальнейшие колонки уже содержут эту информацию

In [ ]:
df = df.drop(labels=["reviews"], axis=1)

## Мусорные данные Supported languages, Full audio languages

In [ ]:
df["supported_languages"] = df["supported_languages"].str.replace(r"[\[\]'\"]", "", regex=True).apply(
    lambda x: len([i for i in str(x).split(',') if i.strip()]))

In [ ]:
df["full_audio_languages"] = df["full_audio_languages"].str.replace(r"[\[\]'\"]", "", regex=True).apply(
    lambda x: len([i for i in str(x).split(',') if i.strip()]))

In [ ]:
df[["supported_languages", "full_audio_languages"]]

## Данные Estimated owners 

In [ ]:
df["estimated_owners"] = df["estimated_owners"].apply(lambda x: sum([int(i.strip()) for i in x.split('-') if i.strip()])/2)

In [ ]:
df["estimated_owners"].describe()

## Данные Release date

In [ ]:
df["release_date"] = pd.to_datetime(df["release_date"])

df["release_year"] = df["release_date"].dt.year
df["release_month"] = df["release_date"].dt.month

df["release_date"]

In [ ]:
time_ = time.localtime()
time_format = pd.to_datetime(f"{time_.tm_year}-{time_.tm_mon}-{time_.tm_wday}")
df["days_since_release"] = (time_format - df["release_date"]).dt.days

In [ ]:
df["days_since_release"]

In [ ]:
df = df.drop(["release_date"], axis=1)

## Данные Website, Support url, Support email, Windows, Mac, Linux, Metactritic score, Metacritic url


In [ ]:
df["website"] # стоит удалить

In [ ]:
df[["support_url", "support_email"]]# стоит удалить

In [ ]:
df = df.drop(["support_url", "support_email", "website"], axis=1)

In [ ]:
df["windows"] # проще считать что все играют на win, >97% на нем играют

In [ ]:
df = df.drop(["windows","mac","linux"],axis=1)

In [ ]:
df.info()

## Данные Required age, About the game, Achievements 

In [ ]:
df["required_age"] = df["required_age"].apply(lambda x: 1 if x >=18 else 0)
#пока решил оставить

In [ ]:
df["about_the_game"] # все это же содержится в positive и negative

In [ ]:
df = df.drop(["about_the_game"], axis=1)

In [ ]:
df = df.drop(["achievements"],axis=1) # их просто слииишком мало
# да и кол во достижений ниуву не коррелирует

In [ ]:
df["user_score"].describe() #данные разряжены

In [ ]:
df = df.drop(["user_score"], axis=1)

In [ ]:
df = df.drop(["score_rank"], axis=1)

In [ ]:
df["metacritic_url"] = df["metacritic_url"].notna().astype(int)

## Positive, Negative, Developers, Сategories, Publishers

In [ ]:
df["positive"].describe()

In [ ]:
df["negative"].describe()

In [ ]:
df["persent_positive"] = df["positive"]/(df["positive"] + df["negative"])

In [ ]:
df["persent_positive"] = df["persent_positive"].fillna(0)
df["persent_positive"]

In [ ]:
# все средние и медианные времена игр оставляю

In [ ]:
df["developers"]

In [ ]:
df["developers"] = df["developers"].str.replace("[", "").str.replace("]", "").str.replace(" ", "")
df["developers"] = df["developers"].replace('', np.nan)

In [ ]:
df['developers'] = df['developers'].fillna('Unknown')
top_devs = df['developers'].value_counts().nlargest(10).index
df['developers_grouped'] = df['developers'].apply(lambda x: x if x in top_devs else 'Other')
df["developers_grouped"].value_counts()

In [ ]:
dev_dummies = pd.get_dummies(df['developers_grouped'], prefix='dev')
df = pd.concat([df, dev_dummies], axis=1)
df = df.drop(["developers", "developers_grouped"], axis=1)

In [ ]:
df = df.drop(["dev_Other", "dev_Unknown"], axis=1)

In [ ]:
df["publishers"] = df["publishers"].str.replace("[", "").str.replace("]", "").str.replace(" ", "")
df["publishers"] = df["publishers"].replace('', np.nan)
df['publishers'] = df['publishers'].fillna('Unknown')

top_devs = df['publishers'].value_counts().nlargest(10).index
df['publishers_grouped'] = df['publishers'].apply(lambda x: x if x in top_devs else 'Other')
df["publishers_grouped"].value_counts()

dev_dummies = pd.get_dummies(df['publishers_grouped'], prefix='pub')
df = pd.concat([df, dev_dummies], axis=1)
df = df.drop(["publishers", "publishers_grouped"], axis=1)

df = df.drop(["pub_Other", "pub_Unknown"], axis=1)

##  Муссорные данные: Tags, Screenshots, Movies, Packges, Notes

In [ ]:
df["movies"].value_counts() #пон

In [ ]:
df["tags"]

In [ ]:
df = df.drop(["tags", "movies", "packages", "screenshots", "notes"], axis=1)

In [ ]:
len(df.columns)

## Short_description и Detailed_description

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA

In [ ]:
model = SentenceTransformer("all-MiniLM-l6-v2")

In [ ]:
embendings = model.encode(df["short_description"].astype(str).tolist(), batch_size=64, show_progress_bar=True)

In [ ]:
embendings.shape

In [ ]:
n_comp = 5
pca = PCA(n_components=n_comp)
pca_features = pca.fit_transform(embendings)

In [ ]:
pca_cols = [f'name_desc_{i}' for i in range(n_comp)]
df_pca = pd.DataFrame(pca_features, columns=pca_cols, index=df.index)

df = pd.concat([df, df_pca], axis=1)

In [ ]:
df = df.drop(["detailed_description"], axis=1)

In [ ]:
df = df.drop(["short_description"], axis=1)

## Genres и categories

In [ ]:
df["genres"]

In [ ]:
df["categories"]

In [ ]:
import ast

In [ ]:
df['genres'] = df['genres'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

In [ ]:
top_five = df["genres"].explode().value_counts().nlargest(5).index.to_list()
top_five

In [ ]:
for genre in top_five:
    
    df[f'genre_{genre}'] = df['genres'].apply(lambda x: 1 if genre in x else 0)

In [ ]:
df['categories'] = df['categories'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

In [ ]:
#temp_df = df[['categories', 'peak_ccu']].explode('categories')
#genre_stats = temp_df.groupby('categories')['peak_ccu'].agg(['mean', 'count'])

#top_5_by_online = genre_stats[genre_stats['count'] > 50] \
 #   .nlargest(5, 'mean').index.tolist()

In [ ]:
top_5 = df['categories'].explode().value_counts().nlargest(5).index.to_list()

In [ ]:
#temp_df.groupby('categories')['peak_ccu'].agg(['mean', 'count'])

In [ ]:
#genre_stats[genre_stats['count'] > 50] \
 #   .nlargest(5, 'mean').index.tolist()

In [ ]:
for cat in top_5:
    
    df[f'cat_{cat}'] = df['categories'].apply(lambda x: 1 if genre in x else 0)

In [ ]:
df = df.drop(["genres", "categories"], axis=1)

## Проверка наличия дубликатов

In [ ]:
duplicate_count = df.duplicated().sum()

In [ ]:
duplicate_count

In [ ]:
df = df.drop_duplicates()

In [ ]:
duplicate_count = df.duplicated().sum()

In [ ]:
duplicate_count

In [ ]:
df.columns

## Дополнительный штрих

In [ ]:
df['peak_ccu_log'] = np.log1p(df['peak_ccu'])

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
df.to_csv("../Data/complete_2_lab.csv")

In [ ]:
len(df.columns)

## Еще штрихи

In [ ]:
df["price"].describe()

In [ ]:
import seaborn as sns

In [ ]:
sns.boxplot(df["price"])

In [ ]:
df[df["price"] <= 60] # вроде не подходит

In [ ]:
df = df[df["price"] <= 60]

In [ ]:
sns.boxplot(df["recommendations"]) # можно логарифмировать

In [ ]:
df["recommendations"] = np.log1p(df["recommendations"])

In [ ]:
df["recommendations"].describe()

In [ ]:
sns.boxplot(df["dlc_count"])

In [ ]:
df[df["dlc_count"] <= 100] # вроде подходит

In [ ]:
df = df[df["dlc_count"] <= 100]

In [ ]:
df["positive"] = np.log1p(df["positive"])

In [ ]:
df["negative"] = np.log1p(df["negative"])

In [ ]:
sns.boxplot(df["average_playtime_2weeks"]) # эта и 3 похожие колонки спорны крайне

In [ ]:
# я хочу предсказывать онлайн и для новых игр тоже
df["average_playtime_forever"] = np.log1p(df["average_playtime_forever"])
df["average_playtime_2weeks"] = np.log1p(df["average_playtime_2weeks"])
df["median_playtime_forever"] = np.log1p(df["median_playtime_forever"])
df["median_playtime_2weeks"] = np.log1p(df["median_playtime_2weeks"])

In [ ]:
#переделать и сохранить)
